# SIAL Algolia 검색 요청 재현 (API key 자동 확보)

브라우저에서 관측된 요청:

```
POST https://w1bghm6ujn-3.algolianet.com/1/indexes/*/queries?x-algolia-agent=...
```

## API key 는 어디서 오는가

SIAL 전시회 카탈로그는 Comexposium `connect2` Vue 앱이다.
이 앱은 하드코딩된 키를 쓰지 않고, 부팅 시 GraphQL 로 설정을 받아온다.

```
www.sialparis.com/en/exhibitors-2026
  └ connect2.prod.comexposium-webservices.com/connect2Loader.js
      └ js/cxpmc2.*.js  →  POST https://api.comexposium-sso.com/_/graphql
          query { Exhibitions { exhibition(id:"sial") {
                    embedConfigTemplate { globalConfigResult } } } }
          → globalConfigResult.algoliaConfig.{applicationId, apiKey}
```

따라서 `requests.Session` 으로 이 GraphQL 만 한 번 치면 `applicationId` / `apiKey`
(검색 전용 public key) 와 인덱스 이름 목록까지 전부 얻을 수 있다. 수동 입력 불필요.

In [ ]:
import json
import time
from urllib.parse import urlencode

import requests

## 1. 상수 + 세션

In [ ]:
GRAPHQL_URL = "https://api.comexposium-sso.com/_/graphql"
SITE = "https://www.sialparis.com"
EXHIBITION_ID = "sial"
LOCALE = "en"          # "fr" 로 바꾸면 프랑스어 인덱스

ALGOLIA_AGENT = (
    "Algolia for JavaScript (4.26.0); Browser; instantsearch.js (4.95.0); "
    "Vue (3.5.33); Vue InstantSearch (4.25.1); JS Helper (3.28.2)"
)

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
)

# 캡처된 요청에 있던 값 그대로.
# 주의: 이 중 "made_in" 과 "specs.made_in_france.value" 는 이 인덱스의
# attributesForFaceting 에 없어서 응답 facets 에 아예 나타나지 않는다 (Algolia 가 조용히 무시).
# 원산지 패싯의 실제 속성명은 "madeIn" 이다. (아래 fetch_by_facet 참고)
FACETS = [
    "brand.name",
    "businessArea.categories.lvl0",
    "certifications.label",
    "exhibitor.name",
    "made_in",
    "productTypes",
    "specs.made_in_france.value",
    "stands.sector",
    "thematics",
]
OPTIONAL_FILTERS = [
    "orders.list_top_position_1:true<score=10000>",
    "orders.list_top_position_2:true<score=9000>",
]


def make_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
        "Accept-Language": f"{LOCALE}-US,{LOCALE};q=0.9",
        "Origin": SITE,
        "Referer": SITE + "/",
    })
    return s


SESSION = make_session()
SESSION

## 2. API key 자동 확보

Vue 앱이 실제로 보내는 것과 동일한 GraphQL 쿼리 / 헤더를 사용한다.

In [ ]:
CONFIG_QUERY = """
query ($exhibitionId: ID!) {
  Exhibitions {
    exhibition(id: $exhibitionId) {
      embedConfigTemplate {
        globalConfigResult
      }
    }
  }
}
"""


def fetch_global_config(session, exhibition_id=EXHIBITION_ID, locale=LOCALE, timeout=30):
    """connect2 앱이 부팅 시 하는 것과 같은 설정 조회."""
    r = session.post(
        GRAPHQL_URL,
        headers={
            "Content-Type": "application/json",
            "gql-exhibitionContext": json.dumps({"id": exhibition_id}),
            "gql-translationLocale": locale,
        },
        json={"query": CONFIG_QUERY, "variables": {"exhibitionId": exhibition_id}},
        timeout=timeout,
    )
    r.raise_for_status()
    data = r.json()
    if data.get("errors"):
        raise RuntimeError(f"GraphQL 오류: {data['errors']}")
    return data["data"]["Exhibitions"]["exhibition"]["embedConfigTemplate"]["globalConfigResult"]


def fetch_algolia_config(session, **kw):
    cfg = fetch_global_config(session, **kw)["algoliaConfig"]
    if not cfg.get("applicationId") or not cfg.get("apiKey"):
        raise RuntimeError("algoliaConfig 에 applicationId/apiKey 가 없음")
    return cfg


ALGOLIA_CONFIG = fetch_algolia_config(SESSION)

APP_ID = ALGOLIA_CONFIG["applicationId"]
API_KEY = ALGOLIA_CONFIG["apiKey"]           # 검색 전용 public key

print("applicationId :", APP_ID)
print("apiKey        :", API_KEY)
print("기타 앱       :", {k: v for k, v in ALGOLIA_CONFIG.items()
                          if isinstance(v, str) and k not in ('applicationId', 'apiKey')})

In [ ]:
# 확보한 키를 세션 기본 헤더에 심는다 → 이후 요청은 헤더 신경 쓸 필요 없음
SESSION.headers.update({
    "x-algolia-api-key": API_KEY,
    "x-algolia-application-id": APP_ID,
})

# 캡처된 호스트는 -3 (DSN 리트라이 호스트). 앞쪽부터 순서대로 시도한다.
HOSTS = [
    f"{APP_ID.lower()}-dsn.algolia.net",
    f"{APP_ID.lower()}-1.algolianet.com",
    f"{APP_ID.lower()}-2.algolianet.com",
    f"{APP_ID.lower()}-3.algolianet.com",
]
print(HOSTS)

## 3. 인덱스 이름도 설정에서 가져온다

`algoliaConfig.search.sorts.<종류>.<정렬id>.value` 안에 `$LOCALE$` 플레이스홀더가 들어있다.

In [ ]:
SORTS = ALGOLIA_CONFIG["search"]["sorts"]


def index_name(kind="products", sort="default", locale=LOCALE):
    return SORTS[kind][sort]["value"].replace("$LOCALE$", locale)


for kind, entries in SORTS.items():
    print(f"[{kind}]")
    for sort_id, entry in entries.items():
        label = (entry.get("label") or {}).get(LOCALE, "")
        print(f"   {sort_id:<38} {entry['value'].replace('$LOCALE$', LOCALE)}  {label}")

INDEX = index_name("products")
print("\n사용할 인덱스:", INDEX)   # -> catalog.prod.sial.products.en (캡처와 동일)

## 4. 요청 빌더 + 검색

`params` 는 URL 인코딩된 쿼리스트링 문자열이다 (Algolia multi-queries 규격).

In [ ]:
def build_params(query="", page=0, hits_per_page=None, facet_filters=None,
                 filters=None, extra=None):
    """캡처된 요청과 동일한 params 문자열을 만든다."""
    p = {
        "facets": json.dumps(FACETS, separators=(",", ":")),
        "highlightPostTag": "__/ais-highlight__",
        "highlightPreTag": "__ais-highlight__",
        "maxValuesPerFacet": 1501,
        "optionalFilters": json.dumps(OPTIONAL_FILTERS, separators=(",", ":")),
        "page": page,
        "query": query,
    }
    if hits_per_page is not None:
        p["hitsPerPage"] = hits_per_page
    if facet_filters:                      # 예: [["madeIn:France"], ["brand.name:X"]]
        p["facetFilters"] = json.dumps(facet_filters, separators=(",", ":"))
    if filters:                            # 예: 'orders.search_page_sponsored:true'
        p["filters"] = filters
    if extra:                              # 위 키를 덮어쓸 수 있음 (예: facets 교체)
        p.update(extra)
    return urlencode(p)


def search(query="", page=0, index=None, session=None, timeout=20, **params_kw):
    """단일 검색. 호스트가 실패하면 다음 DSN 으로 넘어간다."""
    s = session or SESSION
    payload = {
        "requests": [{
            "indexName": index or INDEX,
            "params": build_params(query, page, **params_kw),
        }]
    }
    last = None
    for host in HOSTS:
        try:
            r = s.post(
                f"https://{host}/1/indexes/*/queries",
                params={"x-algolia-agent": ALGOLIA_AGENT},
                json=payload,
                timeout=timeout,
            )
            r.raise_for_status()
            return r.json()["results"][0]
        except requests.RequestException as e:
            last = e
            continue
    raise last


print(build_params()[:260], "...")

## 5. 실행 (캡처와 동일한 조건: query="", page=0)

In [ ]:
res = search(query="", page=0)

print("nbHits      :", res.get("nbHits"))
print("nbPages     :", res.get("nbPages"))
print("hitsPerPage :", res.get("hitsPerPage"))
print("processingMS:", res.get("processingTimeMS"))
print("hits        :", len(res.get("hits", [])))

In [ ]:
hits = res.get("hits", [])
for h in hits[:10]:
    print(f"- {h.get('name')}  |  {(h.get('exhibitor') or {}).get('name')}  |  {h.get('madeIn')}")

print()
print(json.dumps(hits[0], ensure_ascii=False, indent=2)[:2000] if hits else "no hits")

In [ ]:
for name, values in (res.get("facets") or {}).items():
    top = sorted(values.items(), key=lambda kv: -kv[1])[:5]
    print(f"{name} ({len(values)}) -> {top}")

## 6. 전체 페이지 수집

Algolia 는 `page` 기반 페이징에서 `paginationLimitedTo` (여기선 약 2000건) 까지만 돌려준다.
`nbHits` 가 그보다 크면 패싯(`facetFilters`)으로 쪼개서 여러 번 받아야 한다.

In [ ]:
def fetch_all(query="", hits_per_page=100, max_pages=None, delay=0.15, **params_kw):
    out = []
    first = search(query, page=0, hits_per_page=hits_per_page, **params_kw)
    out.extend(first["hits"])
    total = first["nbPages"] if max_pages is None else min(first["nbPages"], max_pages)
    for page in range(1, total):
        r = search(query, page=page, hits_per_page=hits_per_page, **params_kw)
        out.extend(r["hits"])
        print(f"page {page + 1}/{total} - 누적 {len(out)}", end="\r")
        time.sleep(delay)
    print()
    return out


# all_hits = fetch_all(max_pages=3)
# len(all_hits)

In [ ]:
def fetch_by_facet(facet="madeIn", query="", hits_per_page=100, delay=0.15):
    """패싯 값별로 나눠서 2000건 상한을 우회하고 수집한다 (objectID 기준 중복 제거).

    facet 값이 하나도 없는 상품(예: madeIn 미기입)은 잡히지 않으므로
    마지막에 nbHits 와 수집량을 비교해 누락분을 확인할 것.
    """
    # 분할 기준 패싯은 FACETS 에 없을 수도 있으니 명시적으로 요청한다
    only = {"facets": json.dumps([facet], separators=(",", ":"))}
    base = search(query, page=0, hits_per_page=1, extra=only)
    values = sorted((base.get("facets") or {}).get(facet, {}).items(), key=lambda kv: -kv[1])
    if not values:
        raise RuntimeError(f"'{facet}' 는 이 인덱스의 패싯이 아니다 (attributesForFaceting 확인)")
    print(f"{facet}: {len(values)} 개 값 / 전체 {base['nbHits']}건 "
          f"/ 패싯 합계 {sum(c for _, c in values)}건")

    seen, out = set(), []
    for value, count in values:
        hits = fetch_all(query, hits_per_page=hits_per_page, delay=delay,
                         facet_filters=[[f"{facet}:{value}"]])
        new = [h for h in hits if h["objectID"] not in seen]
        seen.update(h["objectID"] for h in new)
        out.extend(new)
        print(f"  {value}: 예상 {count} / 수신 {len(hits)} / 누적 {len(out)}")
    return out


# all_hits = fetch_by_facet("madeIn")     # 최대 값이 ~700건이라 2000 상한에 안 걸린다
# len(all_hits)

## 7. 저장

In [ ]:
# with open("sial_products.json", "w", encoding="utf-8") as f:
#     json.dump(all_hits, f, ensure_ascii=False, indent=2)

# import pandas as pd
# df = pd.json_normalize(all_hits)
# df.to_csv("sial_products.csv", index=False, encoding="utf-8-sig")
# df.head()